In [3]:
import mlrun

In [4]:
project = mlrun.get_or_create_project("kedro-test")

> 2025-12-01 15:12:11,100 [info] Project loaded successfully: {"project_name":"kedro-test"}


In [22]:
# generate user data
function = project.set_function(name="generate-data", func="../generate_data.py", requirements=["pandas", "numpy"], kind="job", image="mlrun/mlrun:1.10.0-rc41", handler="main")

In [23]:
function.deploy()

> 2025-11-28 14:12:40,724 [info] Started building image: .mlrun/func-kedro-test-generate-data:latest
INFO[0000] Retrieving image manifest datanode-registry.iguazio-platform.app.vmdev93.lab.iguazeng.com:80/quay.io/mlrun/mlrun:1.10.0-rc41 
INFO[0000] Retrieving image datanode-registry.iguazio-platform.app.vmdev93.lab.iguazeng.com:80/quay.io/mlrun/mlrun:1.10.0-rc41 from registry datanode-registry.iguazio-platform.app.vmdev93.lab.iguazeng.com:80 
INFO[0000] Built cross stage deps: map[]                
INFO[0000] Retrieving image manifest datanode-registry.iguazio-platform.app.vmdev93.lab.iguazeng.com:80/quay.io/mlrun/mlrun:1.10.0-rc41 
INFO[0000] Returning cached image manifest              
INFO[0000] Executing 0 build triggers                   
INFO[0000] Building stage 'datanode-registry.iguazio-platform.app.vmdev93.lab.iguazeng.com:80/quay.io/mlrun/mlrun:1.10.0-rc41' [idx: '0', base-idx: '-1'] 
INFO[0000] Unpacking rootfs as cmd RUN echo 'Installing /empty/requirements.txt...'; cat /

True

In [56]:
function.with_code("../generate_data.py")

In [57]:
function.run()

> 2025-11-28 14:46:00,086 [info] Storing function: {"db":"https://mlrun-api.default-tenant.app.vmdev93.lab.iguazeng.com","name":"generate-data-main","uid":"4dad854218414f1a85f17a5e19f86d41"}
> 2025-11-28 14:46:01,134 [info] Job is running in the background, pod: generate-data-main-n4dmq
Generating synthetic user data...
✓ Generated 1000 samples

Risk Profile Distribution:
{'moderate': 763, 'aggressive': 154, 'conservative': 83}

Sample data (first 5 rows):
      user_id  age  ...  portfolio_volatility  risk_profile
0  USER_00000   56  ...              0.257277      moderate
1  USER_00001   69  ...              0.295327      moderate
2  USER_00002   46  ...              0.117996      moderate
3  USER_00003   32  ...              0.206520      moderate
4  USER_00004   60  ...              0.340437      moderate

[5 rows x 9 columns]

Feature Statistics:
               age         income  ...  risk_tolerance_score  portfolio_volatility
count  1000.000000    1000.000000  ...           1000

project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
kedro-test,...19f86d41,0,Nov 28 14:46:04,2025-11-28 14:46:05.082247+00:00,completed,run,generate-data-main,v3io_user=adminkind=jobowner=adminmlrun/client_version=1.10.0mlrun/client_python_version=3.11.9host=generate-data-main-n4dmq,,,,user_data.csv


> 2025-11-28 14:46:12,847 [info] Run execution finished: {"name":"generate-data-main","status":"completed"}


In [33]:
project.list_artifacts()[0]

{'kind': 'dataset',
 'metadata': {'key': 'user_data_csv',
  'project': 'kedro-test',
  'iter': 0,
  'tree': '8a4d15a7f9c64e99804bd68fe43722de',
  'uid': '50ffca1bb66668b52cf91fdf262260bae90c5934',
  'updated': '2025-11-28 14:19:05.516000+00:00',
  'created': '2025-11-28 14:19:05.516000+00:00',
  'tag': 'latest'},
 'spec': {'src_path': '/mlrun/data/raw/user_data.csv',
  'target_path': 'v3io:///projects/my-project/artifacts/generate-data-main/0/user_data_csv.csv',
  'format': '',
  'has_children': False,
  'producer': {'name': 'generate-data-main',
   'kind': 'run',
   'uri': 'kedro-test/8a4d15a7f9c64e99804bd68fe43722de',
   'owner': 'admin'},
  'license': '',
  'db_key': 'generate-data-main_user_data_csv',
  'parent_uri': None},
 'status': {'state': 'created'},
 'project': 'kedro-test'}

In [18]:
import shutil
from pathlib import Path
project_root = Path("../../kedro/risk-profile-classifier")
shutil.make_archive("archive/riskclassifier", 'zip', root_dir=project_root)

'/Users/Ekaterina_Molchanova/iguazio/mlrun-ark-integration/mlrun/archive/riskclassifier.zip'

In [19]:
artifact = project.log_artifact(
    item="risk-classifier",                                   # ← artifact key
    local_path="archive/riskclassifier.zip",   # ← path to the zip file
    format="zip",
    artifact_path="v3io:///projects/kedro-test/artifacts/",
    upload=True,
)

In [183]:
#dataitem = project.get_artifact('risk-classifier').to_dataitem()
#files = dataitem.download(target_path="archive/test.zip")

In [26]:
# run training
training_function = project.set_function(name="kedro-training", func="mlrun_handler.py", kind="job", image="rokatyy/kedro-mlrun:0.0.1", handler="handler")

In [27]:
training_function.with_code("mlrun_handler.py")

> 2025-12-01 16:42:08,794 [warning] User code exceeds the maximum allowed size of 10000 bytes for non remote source. Consider using `with_source_archive` to add user code as a remote source to the function.


In [28]:
run = training_function.run()

> 2025-12-01 16:42:10,451 [info] Storing function: {"db":"https://mlrun-api.default-tenant.app.vmdev93.lab.iguazeng.com","name":"kedro-training-handler","uid":"1bef7101034e49fbb07b114ec1f6c94b"}
> 2025-12-01 16:42:11,445 [info] Job is running in the background, pod: kedro-training-handler-h5qgw
[12/01/25 16:42:15] INFO     Using                               __init__.py:270
                             '/opt/conda/lib/python3.11/site-pac                
                             kages/kedro/framework/project/rich_                
                             logging.yml' as logging                            
                             configuration.                                     
> 2025-12-01 16:42:16,047 [info] Starting Kedro pipeline: training
> 2025-12-01 16:42:16,047 [info] Project artifact: risk-classifier
> 2025-12-01 16:42:16,047 [info] Data artifact: generate-data-main_user_data.csv
> 2025-12-01 16:42:16,047 [info] Working directory: /tmp/kedro_project
> 2025-12-01 

project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results
kedro-test,...c1f6c94b,0,Dec 01 16:42:15,2025-12-01 16:42:19.346796+00:00,completed,run,kedro-training-handler,v3io_user=adminkind=jobowner=adminmlrun/client_version=1.10.0mlrun/client_python_version=3.11.9host=kedro-training-handler-h5qgw,,,accuracy=0.995aggressive_precision=1.0aggressive_recall=1.0aggressive_f1-score=1.0aggressive_support=31.0conservative_precision=1.0conservative_recall=0.9375conservative_f1-score=0.967741935483871conservative_support=16.0moderate_precision=0.9935064935064936moderate_recall=1.0moderate_f1-score=0.996742671009772moderate_support=153.0macro avg_precision=0.9978354978354979macro avg_recall=0.9791666666666666macro avg_f1-score=0.988161535497881macro avg_support=200.0weighted avg_precision=0.9950324675324675weighted avg_recall=0.995weighted avg_f1-score=0.9949274981611853weighted avg_support=200.0


> 2025-12-01 16:42:23,219 [info] Run execution finished: {"name":"kedro-training-handler","status":"completed"}


In [29]:
run.status.artifacts

[]

In [20]:
serving_function = project.set_function(name="kedro-serving", func="mlrun_handler.py", kind="remote", image="rokatyy/kedro-mlrun:0.0.1", handler="handler")

In [21]:
serving_function.set_env("KEDRO_PIPELINE_NAME", "serving")
serving_function.set_env("DATA_ARTIFACT", "")
serving_function.set_env("KEDRO_PROJECT_ARTIFACT", "risk-classifier")


In [22]:
serving_function.with_code("mlrun_handler.py")

> 2025-12-01 16:14:09,090 [warning] User code exceeds the maximum allowed size of 10000 bytes for non remote source. Consider using `with_source_archive` to add user code as a remote source to the function.


In [23]:
serving_function.deploy()

> 2025-12-01 16:14:10,167 [info] Starting remote function deploy
2025-12-01 16:14:10  (info) Deploying function
2025-12-01 16:14:10  (info) Building
2025-12-01 16:14:11  (info) Staging files and preparing base images
2025-12-01 16:14:11  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2025-12-01 16:14:11  (info) Building processor image
2025-12-01 16:15:59  (info) Build complete
2025-12-01 16:16:08  (info) Function deploy complete
> 2025-12-01 16:16:17,839 [info] Successfully deployed function: {"external_invocation_urls":["kedro-test-kedro-serving.default-tenant.app.vmdev93.lab.iguazeng.com/"],"internal_invocation_urls":["nuclio-kedro-test-kedro-serving.default-tenant.svc.cluster.local:8080"]}


'http://kedro-test-kedro-serving.default-tenant.app.vmdev93.lab.iguazeng.com/'

In [25]:
serving_function.invoke(path="/", body={
  "age": 22,
  "income": 1200,
  "investment_experience_years": 20,
  "savings_ratio": 0.35,
  "debt_ratio": 0.10,
  "risk_tolerance_score": 3,
  "portfolio_volatility": 0.10
})

{'predictions': [{'predicted_risk_profile': 'conservative',
   'probability_aggressive': 0.016666666666666666,
   'probability_conservative': 0.7205545843045843,
   'probability_moderate': 0.26277874902874904,
   'confidence': 0.7205545843045843}]}